# ⚡ Train V-BOT GPT Model on Google Colab GPU with `combined_train.txt`

This notebook trains your custom Decoder-Only Transformer (GPT) language model using your `combined_train.txt` dataset on a free T4 GPU.

---

### Step 1: Enable Free GPU Acceleration
1. In the top menu, go to **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU**.
3. Click **Save**.

In [ ]:
# Verify GPU is active
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU:      {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: Running on CPU. Please switch to T4 GPU under: Runtime > Change runtime type!")

### Step 2: Upload `combined_train.txt`

You can upload your `combined_train.txt` file in either of two ways:
- **Method A (Easiest)**: Click the **Folder icon** 📁 on the left sidebar, then click the **Upload icon** ⬆️ and select `combined_train.txt` from your local project folder.
- **Method B**: Run the cell below to prompt an upload browser dialog.

In [ ]:
import os

# Check if combined_train.txt was uploaded via the left sidebar
if not os.path.exists('combined_train.txt'):
    print("combined_train.txt not found. Opening upload dialog...")
    from google.colab import files
    uploaded = files.upload()
    print(f"Uploaded: {list(uploaded.keys())}")
else:
    print("✅ Found 'combined_train.txt' ready for training!")

with open('combined_train.txt', 'r', encoding='utf-8') as f:
    sample = f.read(300)
print("\n--- First 300 characters of dataset ---")
print(sample)
print("---------------------------------------")

### Step 3: Model Architecture & Tokenizer Setup
*(Scaled GPU configuration: 256 embedding dimension, 6 layers, 6 attention heads, 128 context window)*

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Reproducibility & Device
torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ==============================================================================
# Hyperparameters (Optimized for ~138k char dataset on T4 GPU)
# ==============================================================================
batch_size    = 64      # Process 64 parallel sequence chunks per batch
block_size    = 128     # Context window length (128 characters)
max_iters     = 3000    # Training iterations (~2-3 minutes on T4 GPU)
eval_interval = 300     # Log training & validation loss every 300 steps
learning_rate = 5e-4    # Learning rate for AdamW
eval_iters    = 50      # Batches to average during loss evaluation
n_embd        = 256     # Embedding dimension
n_head        = 6       # Number of self-attention heads
n_layer       = 6       # Number of stacked transformer blocks
dropout       = 0.2     # Regularization dropout rate

# ==============================================================================
# 1. Load Dataset & Character-Level Tokenizer
# ==============================================================================
with open('combined_train.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read().replace('\r\n', '\n').replace('\r', '')

chars = sorted(list(set(raw_text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda l: ''.join([itos[i] for i in l])

data_tensor = torch.tensor(encode(raw_text), dtype=torch.long)
split_idx = int(0.9 * len(data_tensor))
train_data = data_tensor[:split_idx]
val_data   = data_tensor[split_idx:]

print(f"Vocabulary Size: {vocab_size} unique characters")
print(f"Train tokens:    {len(train_data):,} tokens")
print(f"Val tokens:      {len(val_data):,} tokens")
print(f"Vocabulary:      {''.join(chars)}")

# ==============================================================================
# 2. Batch Generator
# ==============================================================================
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# ==============================================================================
# 3. Transformer Neural Architecture (Exact V-BOT Implementation)
# ==============================================================================
class Head(nn.Module):
    """Single Head of Scaled Dot-Product Self-Attention with Causal Mask"""
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = (q @ k.transpose(-2, -1)) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ v

class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention in Parallel"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    """Position-wise 2-layer MLP with GELU non-linearity"""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    """Pre-LayerNorm Transformer Block"""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    """Full Decoder-Only GPT Architecture"""
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[TransformerBlock(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = self.ln_f(self.blocks(tok_emb + pos_emb))
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.7, top_k=10):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-4)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Instantiate Model on GPU
model = GPTLanguageModel(vocab_size=vocab_size).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nInitialized V-BOT Model on {device.upper()}")
print(f"Total Model Parameters: {total_params:,}")

### Step 4: Run GPU Training Loop & Save Checkpoint
*(Takes ~2 to 3 minutes on Google Colab T4 GPU)*

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print(f"Starting GPU Training for {max_iters} iterations...")
print("-" * 60)

for iter_num in range(max_iters):
    if iter_num % eval_interval == 0 or iter_num == max_iters - 1:
        losses = estimate_loss()
        print(f"Iteration {iter_num:4d}/{max_iters} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("-" * 60)
# Save trained weights into model.pt
torch.save(model.state_dict(), 'model.pt')
print("✅ Training Complete! Model checkpoint saved to: model.pt")

### Step 5: Test Conversational Generation
Test prompts from `combined_train.txt` to verify how well V-BOT responds.

In [ ]:
prompts = [
    "User: What is your name?\nAssistant:",
    "User: What is Vishwas AI?\nAssistant:",
    "User: What is PyTorch?\nAssistant:",
    "User: How does a transformer model generate text?\nAssistant:"
]

print("=" * 65)
print("V-BOT INFERENCE TEST SAMPLES")
print("=" * 65)

model.eval()
with torch.no_grad():
    for p in prompts:
        tokens = encode(p)
        context = torch.tensor([tokens], dtype=torch.long, device=device)
        out_ids = model.generate(context, max_new_tokens=150, temperature=0.6, top_k=10)[0].tolist()
        print(f"\n>>> Prompt: {repr(p)}")
        print("-" * 50)
        print(decode(out_ids))
        print("-" * 50)

### Step 6: Download `model.pt` Back to Your Local Workspace
Run this cell to download `model.pt` directly to your browser's Downloads folder. Then move it into your project directory `projects/llm/` to replace the local `model.pt`.

In [ ]:
from google.colab import files
files.download('model.pt')
print("Downloading model.pt...")